**DETECCIÓN Y LECTURA DE MATRÍCULAS EN VÍDEO CON RT-DETRv2 Y PADDLEOCR**

En este cuaderno se detectan personas y vehículos con RT-DETRv2, se les asigna un identificador con ByteTrack, se localiza la matrícula de cada vehículo con el detector de matrículas (RT-DETRv2 tamaño `s`) y se lee su texto con PaddleOCR.

El flujo de trabajo se estructura en:
1. **Configuración inicial**: Importaciones, clonado de RT-DETR y selección del dispositivo (GPU si está disponible).
2. **Modelo de matrículas**: Carga del detector que se usa tanto en la evaluación como en el vídeo.
3. **Funciones de OCR**: Gestión de PaddleOCR, normalización de texto, recorte de matrículas y lectura.
4. **Evaluación del OCR**: Exact match y acierto por carácter sobre imágenes anotadas, con y sin corrección por posición, junto con el techo del OCR y los tiempos de lectura.
5. **Procesamiento de vídeo**: Detección, seguimiento, localización de matrículas y OCR sobre un vídeo, guardando el vídeo anotado y un CSV.

**CONFIGURACIÓN INICIAL: IMPORTS Y DISPOSITIVO**

Se clona el repositorio de RT-DETR si no existe, se añade su ruta a `sys.path` para poder importar `load_rtdetr` y se selecciona la GPU si PyTorch detecta una con CUDA.

In [1]:
import os, sys, subprocess
from functools import partial
from pathlib import Path
import torch

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

#Cargamos el modelo y ponemos la ruta en memoria
if not os.path.exists("RT-DETR"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lyuwenyu/RT-DETR.git"], check=True)


RTDETR_PATH = os.path.abspath("RT-DETR/rtdetrv2_pytorch")
if RTDETR_PATH not in sys.path:
    sys.path.insert(0, RTDETR_PATH)

from load_rtdetr import load_rtdetr
from utils_matriculas import (
    GT_KEYS,
    cargar_gt,
    crear_detector_rtdetr,
    evaluar_dataset,
    get_ocr_paddle,
    leer_placa_paddle,
    procesar_video_rtdetr,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

**MODELO DE MATRÍCULAS**

Se carga el detector de matrículas RT-DETRv2 tamaño `s`, con una única clase `plate`.

In [2]:
PLATE_NAME = {0: "matrícula"}
PLATE_CLASS = PLATE_NAME.keys()
PLATE_COLOR = {0: (128, 255, 128)}

model_plate, postprocessor_plate = load_rtdetr(family="v2", size="s", device=DEVICE)

Cargando RT-DETR familia 'v2', tamaño 's' en cuda


**FUNCIONES DE OCR: PADDLEOCR**

Se inicializa PaddleOCR una sola vez `get_ocr_paddle` y se fija la función de lectura `leer_placa`.:
- **`norm`**: Normaliza el texto a mayúsculas alfanuméricas.
- **`lev`**: Distancia de Levenshtein entre dos cadenas, para el acierto por carácter.
- **`recortar`**: Recorta una región con margen porcentual o en píxeles, ajustada a los límites de la imagen.
- **`leer_placa_paddle`**: Reescala el recorte, lo lee con PaddleOCR y devuelve el texto y la confianza media.
- **`corregir`**: Corrige la lectura según la posición de cada carácter en la matrícula española.

In [ ]:
# Inicializamos el OCR globalmente (PaddleOCR se crea solo la primera vez; None si no esta instalado)
ocr_global = get_ocr_paddle()

# leer_placa_paddle (utils) devuelve (texto normalizado, confianza media del OCR)
leer_placa = partial(leer_placa_paddle, ocr_inst=ocr_global)

**EVALUACIÓN DEL OCR SOBRE EL DATASET**

Se evalúa el conjunto detector + OCR sobre las imágenes anotadas, comparando contra el texto real. Se obtienen:
- **Exact match**: Proporción de matrículas leídas exactamente igual que la anotación.
- **Acierto por carácter**: Similitud media basada en la distancia de Levenshtein.
- **Techo del OCR**: Lectura directa del recorte de la anotación, sin depender del detector.
- **Lectura cruda vs. corregida**: La lectura tal cual y tras la corrección por posición.
- **Tiempos**: Milisegundos por lectura (media y mediana), con una lectura de calentamiento previa.

La función `evaluar_dataset` está en `utils_matriculas.py` y es la misma en todos los cuadernos: es independiente del detector, que se le pasa como una función `detectar(img) -> [(caja_xyxy, confianza)]`.

Aquí solo se construye el detector con `crear_detector_rtdetr`, que adapta RT-DETR a lo que espera `evaluar_dataset`. Aplica **letterbox**, deshace el letterbox para devolver las cajas en píxeles originales y usa el orden `(w, h)` que espera el postprocesador. Ya no hace falta convertir la imagen a RGB ni gestionar `no_grad` aquí.

Los resultados se guardan en `ocr_eval/rt-detr_resultados_ocr.csv` y los recortes mal leídos en `ocr_eval/errores-rtdetr`.

In [ ]:
# Configuración de rutas y parámetros del dataset
BASE = Path("plates/raw-dataset")
IMAGENES = BASE / "images"
ANOTACIONES = BASE / "annotations"
SPLIT = "train"
MAX_IMAGENES = 200     # None para evaluar el dataset completo
CONF_METRICAS = 0.35
IOU_MATCH = 0.5
PAD = 0.05
EXTS = {".jpg", ".jpeg", ".png"}

SALIDA = Path("ocr_eval")

# cargar_gt (utils) recibe la carpeta de anotaciones y las claves como argumentos: las fijamos aqui
gt_dataset = partial(cargar_gt, anotaciones_dir=ANOTACIONES, gt_keys=GT_KEYS)

# Detector RT-DETR de matrículas con letterbox: imagen BGR -> [(caja_xyxy, confianza), ...]
detectar_rtdetr = crear_detector_rtdetr(model_plate, postprocessor_plate, DEVICE,
                                        CONF_METRICAS, PLATE_CLASS)


# Ejecución de la evaluación de métricas
if IMAGENES.exists():
    lista = sorted(p for p in IMAGENES.iterdir() if p.suffix.lower() in EXTS)
    if MAX_IMAGENES:
        lista = lista[:MAX_IMAGENES]
    evaluar_dataset(lista, detectar_rtdetr, leer_placa, gt_dataset, SALIDA, "errores-rtdetr",
                    "rt-detr_resultados_ocr.csv", "RT-DETR", iou_match=IOU_MATCH, pad=PAD,
                    gt_keys=GT_KEYS, delimiter=";")
else:
    print(f"Directorio {IMAGENES} no encontrado. Comprueba la ruta del dataset.")

**PARÁMETROS DEL VÍDEO Y MODELO DE VEHÍCULOS**

Parámetros de detección, seguimiento y OCR sobre vídeo:
- **Clases y colores**: Personas, coches, motos, autobuses y camiones (clases COCO).
- **Recortes de vehículo**: Las matrículas se buscan dentro del recorte de cada vehículo y heredan su `tracker_id`.
- **Lectura OCR**: Frecuencia, número máximo de intentos y confianza mínima para dar una lectura por buena.
- **Patrones de matrícula**: Las expresiones regulares de formatos europeos `PATRONES_MATRICULA` y `matricula_valida`.

Aquí se carga también el detector de vehículos y personas RT-DETRv2 tamaño `l`.

In [ ]:
CLASS_NAMES = {0: 'persona', 2: 'coche', 3: 'moto', 5: 'guagua', 7: 'camión'}
BASE_CLASS = CLASS_NAMES.keys()
BASE_COLORS = {0: (0, 255, 0), 2: (0, 0, 255), 3: (255, 0, 255), 5: (0, 255, 255), 7: (128, 128, 255)}

# Las matriculas se buscan dentro del recorte de cada vehiculo y heredan su tracker_id
MAX_CROPS = 8          # maximo de vehiculos por frame donde se buscan matriculas
MIN_CROP_SIDE = 48     # px: vehiculos mas pequenos que esto se ignoran
CROP_MARGIN = 0.05     # margen extra alrededor del vehiculo
PLATE_CONF = 0.3      # confianza minima del detector de matriculas
HOLD_FRAMES = 5        # frames que se mantiene la caja de matricula si no se detecta

# Parámetros visuales y de OCR de matrículas
COLOR_PLACA = (0, 165, 255)
ESCALA_TEXTO_PLACA = 0.35

# Parámetros del OCR sobre vídeo
OCR_CADA = 5
MIN_ANCHO_OCR = 40
MIN_CHARS_PLACA = 4
OCR_SCORE_OK = 0.5
OCR_MAX_INTENTOS = 6

model, postprocessor = load_rtdetr(family="v2", size="l", device=DEVICE)

**TRACKING Y DETECCIÓN RT-DETR**

Las funciones del procesamiento de vídeo están en `utils_matriculas.py` y las utiliza `procesar_video_rtdetr`:
- **`preprocess`**: Redimensiona el fotograma a 640x640 y lo convierte en tensor, una sola vez por fotograma.
- **`make_tracker`**: Crea un tracker ByteTrack con el tiempo de memoria indicado.
- **`letterbox`**: Redimensiona el recorte manteniendo la proporción y rellena con gris, para no deformar las matrículas.
- **`run_detector`**: Ejecuta el detector y filtra por confianza y clases.
- **`detect_plates_in_crops`**: Busca la mejor matrícula dentro del recorte de cada vehículo y la devuelve en coordenadas del fotograma.

**PROCESAMIENTO DE VÍDEO**

Se recorre el vídeo fotograma a fotograma: se detectan personas y vehículos, se les asigna un identificador, se busca la matrícula de cada vehículo y se lee con PaddleOCR. El bucle completo está en `procesar_video_rtdetr`. Se guardan el vídeo anotado `resultado.mp4` y un CSV con todos los resultados `resultado.csv` en `RT-DETR_PaddleOCR_result`.

In [ ]:
path_test = 'C0142.MP4'
output_dir = os.path.abspath('RT-DETR_PaddleOCR_result')

plates_seen, lecturas = procesar_video_rtdetr(
    path_test, output_dir, model, postprocessor, model_plate, postprocessor_plate,
    DEVICE, leer_placa, CLASS_NAMES, BASE_COLORS,
    plate_classes=PLATE_CLASS,
    conf_vehiculos=0.5,
    plate_conf=PLATE_CONF, hold_frames=HOLD_FRAMES, max_crops=MAX_CROPS,
    min_crop_side=MIN_CROP_SIDE, crop_margin=CROP_MARGIN,
    color_placa=COLOR_PLACA, escala_texto_placa=ESCALA_TEXTO_PLACA,
    ocr_cada=OCR_CADA, min_ancho_ocr=MIN_ANCHO_OCR, min_chars_placa=MIN_CHARS_PLACA,
    ocr_score_ok=OCR_SCORE_OK, ocr_max_intentos=OCR_MAX_INTENTOS,
)